# Avoiding full dense reconstruction
Profiling the kl_factor_update_largedims function has shown that memory keeps blowing up at
We will try to avoid that here by only reconstructing an explicit sparse copy

In [56]:
import tensorly as tl
import pytensorlab as ptl

from tensorly.tucker_tensor import validate_tucker_rank
from tensorly.tenalg import mode_dot

from tensormet.sparse_ops import (
    unfold_from_vectorized_sparse,
    ptl_tucker_to_tensor,
)
from tensormet.utils import ThreadBudget, DATA_DIR, compute_num_threads, select_gpu
from tensormet.tucker_tensor import SparseTupleTensor
from tensormet.config import ExperimentConfig, TrainingConfig, EvalConfig, RunConfig
from tensormet.sparse_ops import initialize_nonnegative_tucker

from memory_profiler import profile
import os
import pickle
from pathlib import Path
import time
import numpy as np

device = select_gpu()

tl.set_backend("cupy")

Selected GPU 1 with 10397.94 MB used memory.


In [117]:
dataset = "fineweb-en"
dim = 8000
method = "siiSoftPlus"
divergence = "kl"
name = "profile"
random_state = 1
rank = [500, 500, 500]
max_cpu_frac = 0.66
iters = 500
tol = 1e-5
patience = 5
rec_check_every = 1
rec_log_every = 1
sem_check_every = 5
sem_error_type = "absolute_prob_score"
max_cpu_frac = 0.66
thread_budget = ThreadBudget(n_threads=compute_num_threads(max_cpu_frac))

In [118]:
vocab_path = os.path.join(DATA_DIR, "tensors", dataset, f"vocabularies/{dim}.pkl")
with open(vocab_path, "rb") as f:
    vocab = pickle.load(f)
cfg = RunConfig(
    exp=ExperimentConfig(
        dataset=dataset,
        method=method,
        divergence=divergence,
        dim=dim,
        rank=tuple(rank),
        name=name,
        random_state=random_state,
        max_cpu_frac=max_cpu_frac,
        data_dir=Path(DATA_DIR),
    ),
    train=TrainingConfig(
        n_iter_max=iters,
        tol=tol,
        epsilon=1e-12,
        warmup_steps=1 if divergence == "kl" else 50,
        patience=patience,
        normalize_factors=False,
        verbose=True,
        return_errors="full",
    ),
    eval=EvalConfig(
        rec_log_every=rec_log_every,
        rec_check_every=rec_check_every,
        sem_check_every=sem_check_every,
        sem_error_type=sem_error_type,
        remove_OOV=False,
    ),
)
print(cfg)
paths = cfg.artifact_paths()
for p in paths.values():
    if isinstance(p, Path):
        p.parent.mkdir(parents=True, exist_ok=True)



RunConfig(exp=ExperimentConfig(dataset='fineweb-en', method='siiSoftPlus', divergence='kl', dim=8000, rank=(500, 500, 500), name='profile', random_state=1, max_cpu_frac=0.66, data_dir=PosixPath('/home/local/stefa/data')), train=TrainingConfig(n_iter_max=500, tol=1e-05, epsilon=1e-12, init='random', normalize_factors=False, warmup_steps=1, patience=5, verbose=True, return_errors='full'), eval=EvalConfig(rec_check_every=1, rec_log_every=1, sem_check_every=5, sem_error_type='absolute_prob_score', sem_softmax_temperature=0.1, sem_fitness_target=10000, n_sentence_cache=None, remove_OOV=False))


In [119]:
sparse_tensor = SparseTupleTensor.load_from_disk(
    dataset=dataset,
    method=method,
    dims=dim,
    map_location="cpu",
)
sparse_tensor.tensor_to_sparse("cupy")

In [120]:
shape = tuple(sparse_tensor.shape)
rank = validate_tucker_rank(shape, rank=rank)
modes = list(range(len(rank)))
init = cfg.train.init
core, factors = initialize_nonnegative_tucker(sparse_tensor.tensor, shape, rank, modes, init, random_state)

In [121]:
def kl_factor_update_largedim(vec_tensor, core, factors, mode, shape, thread_budget, epsilon=1e-12):
    """
    One multiplicative KL update for a single factor matrix A_n (for `mode`).
    Use when dimensions become large to fully avoid reconstruction on GPU.

    Parameters
    ----------
    vec_tensor : cupyx.scipy.sparse.coo_matrix
        Vectorized sparse tensor (COO).
    shape : tuple[int, ...]
        Original tensor shape.
    core : cupy.ndarray
        Tucker core on GPU (CuPy).
    factors : list[cupy.ndarray]
        Tucker factors on GPU (CuPy).
    mode : int
        Mode to update.
    epsilon : float
        Small positive constant for numerical stability / nonnegativity.

    Returns
    -------
    A : updated factor
    """
    X = unfold_from_vectorized_sparse(vec_tensor, shape, mode)
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]
    # Dense reconstruction excluding current factor, unfolded along mode
    tucker = ptl.TuckerTensor(core=core_np,
                                  factors=factors_np)
    with thread_budget.limit():
        Z = ptl_tucker_to_tensor(tucker, skip_factor=mode)
    Z = ptl.tens2mat(Z, mode)


    A = factors[mode]  # (I_mode, R)
    rows = X.row
    cols = X.col
    vals = X.data

    # Compute reconstruction only at nonzeros: R_nz = sum_r A[i,r] * Z[r,j]
    # A_rows: (nnz, R)
    A_rows = A[rows, :]

    Z_cols_T = tl.transpose(Z[:, cols.get()])
    Z_cols_T = cp.asarray(Z_cols_T)

    R_nz = tl.sum(A_rows * Z_cols_T, axis=1)
    R_nz = tl.clip(R_nz, a_min=epsilon, a_max=None)


    # W = X / (A Z) at nonzeros

    W_data = vals / R_nz
    W = cpx_sparse.coo_matrix((W_data, (rows, cols)), shape=X.shape)

    numerator = W @ tl.transpose(cp.asarray(Z))


    # denominator = sum_j Z[r,j] broadcast to (I_mode, R)
    # den_row = tl.sum(Z, axis=1)  # (R,)

    den_row = tl.sum(Z, axis=1)
    denominator = den_row[np.newaxis, :]
    denominator = tl.clip(denominator, a_min=epsilon, a_max=None)

    # Multiplicative update
    A = A * (numerator / (cp.asarray(denominator + 1e-12)))
    A = tl.clip(A, a_min=epsilon, a_max=None)
    return A


In [122]:
import cupy as cp

# force CUDA context init
cp.cuda.Device(0).use()
_ = cp.zeros(1)

# basic info
print("CuPy:", cp.__version__)
print("Device:", cp.cuda.runtime.getDeviceProperties(0)["name"].decode())
print("CUDA runtime:", cp.cuda.runtime.runtimeGetVersion())
print("Driver:", cp.cuda.runtime.driverGetVersion())

CuPy: 13.6.0
Device: NVIDIA GeForce RTX 3090
CUDA runtime: 12090
Driver: 12090


In [102]:
import cupyx.scipy.sparse as cpx_sparse
import sparse
mode = 0
start_time = time.time()
factor = kl_factor_update_largedim(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,
                    thread_budget=thread_budget)
print(f"calculated in {time.time()-start_time} seconds")

KeyboardInterrupt: 

In [123]:
import cupyx.scipy.sparse as cpx_sparse

from tensormet.sparse_ops import unfold_from_vectorized_sparse

def kl_factor_update_nodense_equiv(
    vec_tensor,
    core,
    factors,
    mode,
    shape,
    epsilon=1e-12,
    batch_cols=20000,
):
    """
    KL multiplicative update for Tucker factor A^(mode) WITHOUT building dense Z,
    but MATHEMATICALLY EQUIVALENT to the dense-Z version:

        A <- A * ( (W @ Z.T) / sum_j Z[:, j] )

    where W has nonzeros: W_ij = X_ij / (A Z)_ij at the nonzeros of X_(mode).

    Assumes a 3-way tensor (shape = (I0, I1, I2)).
    """
    # Sparse unfolding X_(mode) (I_mode, prod(other dims))
    X = unfold_from_vectorized_sparse(vec_tensor, shape, mode).tocoo()
    rows = X.row
    cols = X.col
    vals = X.data

    A = factors[mode]  # (I_mode, R)
    G = core           # (R, R, R)
    R = A.shape[1]

    # Pick the two "other" factors for this mode, and a decoder for unfolding cols -> (idx1, idx2)
    if mode == 0:
        B, C = factors[1], factors[2]
        I1, I2 = shape[1], shape[2]

        def decode_cols(u):
            j = u // I2
            k = u %  I2
            return j, k

        # Z_u[m, r] = sum_{q,t} G[r,q,t] * B[j_m,q] * C[k_m,t]
        def compute_Zcols(idx1, idx2):
            Bj = B[idx1]  # (m, R)
            Ck = C[idx2]  # (m, R)
            return cp.einsum("rqt,mq,mt->mr", G, Bj, Ck)

        # EXACT denominator: den_row[r] = sum_{j,k} Z[r,(j,k)]
        sB = cp.sum(B, axis=0)  # (R,)
        sC = cp.sum(C, axis=0)  # (R,)
        den_row = cp.einsum("rqt,q,t->r", G, sB, sC)  # (R,)

    elif mode == 1:
        B, C = factors[0], factors[2]
        I0, I2 = shape[0], shape[2]

        def decode_cols(u):
            i = u // I2
            k = u %  I2
            return i, k

        # Z_u[m, q] = sum_{p,t} G[p,q,t] * B[i_m,p] * C[k_m,t]
        def compute_Zcols(idx1, idx2):
            Bi = B[idx1]
            Ck = C[idx2]
            return cp.einsum("pqt,mp,mt->mq", G, Bi, Ck)

        # EXACT denominator: den_row[q] = sum_{i,k} Z[q,(i,k)]
        sB = cp.sum(B, axis=0)
        sC = cp.sum(C, axis=0)
        den_row = cp.einsum("pqt,p,t->q", G, sB, sC)

    else:  # mode == 2
        B, C = factors[0], factors[1]
        I0, I1 = shape[0], shape[1]

        def decode_cols(u):
            i = u // I1
            j = u %  I1
            return i, j

        # Z_u[m, t] = sum_{p,q} G[p,q,t] * B[i_m,p] * C[j_m,q]
        def compute_Zcols(idx1, idx2):
            Bi = B[idx1]
            Cj = C[idx2]
            return cp.einsum("pqt,mp,mq->mt", G, Bi, Cj)

        # EXACT denominator: den_row[t] = sum_{i,j} Z[t,(i,j)]
        sB = cp.sum(B, axis=0)
        sC = cp.sum(C, axis=0)
        den_row = cp.einsum("pqt,p,q->t", G, sB, sC)

    # Match dense version's clipping behavior
    den_row = cp.clip(den_row, a_min=epsilon, a_max=None)
    denominator = den_row[None, :]  # (1, R)

    # Accumulate numerator = W @ Z.T without building full Z:
    # numerator[i, r] = sum_{(i,col) in nnz} (X_ij / (AZ)_ij) * Z[r, col]
    numerator = cp.zeros_like(A)

    # Work over unique columns to reuse Z computations
    ucols, inv = cp.unique(cols, return_inverse=True)

    for start in range(0, int(ucols.size), int(batch_cols)):
        end = min(start + int(batch_cols), int(ucols.size))
        u = ucols[start:end]

        idx1, idx2 = decode_cols(u)
        Z_u = compute_Zcols(idx1, idx2)              # (m, R) == (num_cols_in_batch, R)
        Z_u = cp.clip(Z_u, a_min=epsilon, a_max=None)

        # entries whose column falls in this unique-col batch
        nz_idx = cp.where((inv >= start) & (inv < end))[0]
        if nz_idx.size == 0:
            continue

        r_i = rows[nz_idx]                    # (nnz_b,)
        v_i = vals[nz_idx]                    # (nnz_b,)
        u_i = inv[nz_idx] - start             # (nnz_b,) local col index in this batch

        A_rows = A[r_i]                       # (nnz_b, R)
        Z_rows = Z_u[u_i]                     # (nnz_b, R)

        # (AZ)_nz for those entries
        R_nz = cp.sum(A_rows * Z_rows, axis=1)
        R_nz = cp.clip(R_nz, a_min=epsilon, a_max=None)

        W_data = v_i / R_nz                   # (nnz_b,)

        # scatter-add numerator rows: numerator[r_i] += W * Z
        cp.add.at(numerator, r_i, W_data[:, None] * Z_rows)

    # Multiplicative KL update (match your dense version's structure)
    A_new = A * (numerator / (denominator + 1e-12))
    A_new = cp.clip(A_new, a_min=epsilon, a_max=None)
    return A_new


In [124]:
mode = 0
start_time = time.time()
factor_nodense = kl_factor_update_nodense_equiv(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,)
print(f"calculated in {time.time()-start_time} seconds")


OutOfMemoryError: Out of memory allocating 20,000,000,000 bytes (allocated so far: 10,336,821,760 bytes).

In [105]:
import cupy as cp
from tensormet.sparse_ops import unfold_from_vectorized_sparse


def _unravel_cols_for_mode(cols, shape, mode):
    """
    Convert unfolding column indices -> per-mode indices for all modes != `mode`,
    consistent with the 3-way decoding you used (last remaining mode varies fastest).

    Returns
    -------
    other_modes : list[int]
    idxs       : dict[int, cupy.ndarray]  # maps mode -> (len(cols),) indices
    """
    N = len(shape)
    other_modes = [m for m in range(N) if m != mode]
    other_dims = [shape[m] for m in other_modes]

    u = cols
    idxs_rev = []
    # last remaining mode varies fastest => mod/div in reverse order
    for dim in reversed(other_dims):
        idxs_rev.append(u % dim)
        u = u // dim

    idxs = list(reversed(idxs_rev))
    return other_modes, {m: idxs[i] for i, m in enumerate(other_modes)}


def _einsum_letters(n):
    # enough for typical tensor orders; extend if you ever go beyond 52
    letters = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")
    if n > len(letters):
        raise ValueError(f"Tensor order {n} too large for einsum-letter helper ({len(letters)} available).")
    return letters[:n]


def _tucker_den_row_full(core, factors, mode, epsilon=1e-12):
    """
    Exact denominator vector for KL MU update:
        den_row[r_mode] = sum_over_all_unfolding_columns Z[r_mode, col]
    without forming Z, for arbitrary N-way Tucker.

    core:  (R0, R1, ..., R_{N-1})
    factors[k]: (Ik, Rk)
    """
    N = core.ndim
    letters = _einsum_letters(N)
    core_subs = "".join(letters)

    # s_k[r_k] = sum_i A^{(k)}[i, r_k]
    sums = [cp.sum(factors[k], axis=0) for k in range(N)]
    # einsum: core[a b c ...], sum_b[b], sum_c[c], ... -> output over mode letter
    in_terms = [core_subs] + [letters[k] for k in range(N) if k != mode]
    out_term = letters[mode]
    eq = ",".join(in_terms) + "->" + out_term

    operands = [core] + [sums[k] for k in range(N) if k != mode]
    den_row = cp.einsum(eq, *operands)
    den_row = cp.clip(den_row, a_min=epsilon, a_max=None)
    return den_row


def _compute_Zcols_batch(core, factors, mode, other_modes, idxs_by_mode, epsilon=1e-12):
    """
    Compute Z columns (as rows) for a batch of unfolding columns, without building full Z.

    Returns Z_u with shape (m, R_mode), where m = batch size.
    """
    N = core.ndim
    letters = _einsum_letters(N)
    core_subs = "".join(letters)

    # factor-row matrices for each other mode: (m, Rk)
    mats = [factors[k][idxs_by_mode[k]] for k in other_modes]

    # einsum: core[a b c ...], M_b[m b], M_c[m c], ... -> out[m a_mode]
    in_terms = [core_subs] + [("m" + letters[k]) for k in other_modes]
    out_term = "m" + letters[mode]
    eq = ",".join(in_terms) + "->" + out_term

    Z_u = cp.einsum(eq, core, *mats)
    Z_u = cp.clip(Z_u, a_min=epsilon, a_max=None)
    return Z_u


def kl_factor_update_nodense_equiv_bis(
    vec_tensor,
    core,
    factors,
    mode,
    shape,
    epsilon=1e-12,
    batch_cols=20000,
):
    """
    KL multiplicative update for Tucker factor A^(mode) WITHOUT building dense Z,
    but mathematically equivalent to your dense-Z implementation:

        A <- A * ( (W @ Z.T) / sum_j Z[:, j] )

    where W_ij = X_ij / (A Z)_ij at the nonzeros of X_(mode).

    Works for N-way tensors (N = len(shape) = core.ndim).
    """
    # Sparse unfolding X_(mode)
    X = unfold_from_vectorized_sparse(vec_tensor, shape, mode).tocoo()
    rows = X.row
    cols = X.col
    vals = X.data

    A = factors[mode]  # (I_mode, R_mode)

    # Exact denominator over ALL columns (no approximation)
    den_row = _tucker_den_row_full(core, factors, mode, epsilon=epsilon)
    denominator = den_row[None, :]  # (1, R_mode)

    # Accumulate numerator = W @ Z.T without building full Z
    numerator = cp.zeros_like(A)

    # Reuse computations across repeated columns
    ucols, inv = cp.unique(cols, return_inverse=True)

    # Decode unique columns once per batch (general N-way unravel)
    other_modes = [m for m in range(len(shape)) if m != mode]

    for start in range(0, int(ucols.size), int(batch_cols)):
        end = min(start + int(batch_cols), int(ucols.size))
        u = ucols[start:end]

        _, idxs_by_mode = _unravel_cols_for_mode(u, shape, mode)  # dict: other_mode -> (m,)

        # Z_u: (m, R_mode)
        Z_u = _compute_Zcols_batch(
            core=core,
            factors=factors,
            mode=mode,
            other_modes=other_modes,
            idxs_by_mode=idxs_by_mode,
            epsilon=epsilon,
        )

        # nnz entries belonging to these unique columns
        nz_idx = cp.where((inv >= start) & (inv < end))[0]
        if nz_idx.size == 0:
            continue

        r_i = rows[nz_idx]             # (nnz_b,)
        v_i = vals[nz_idx]             # (nnz_b,)
        u_i = inv[nz_idx] - start      # local [0..m)

        A_rows = A[r_i]                # (nnz_b, R_mode)
        Z_rows = Z_u[u_i]              # (nnz_b, R_mode)

        # (A Z)_nz
        R_nz = cp.sum(A_rows * Z_rows, axis=1)
        R_nz = cp.clip(R_nz, a_min=epsilon, a_max=None)

        W_data = v_i / R_nz            # (nnz_b,)

        # numerator[row] += W * Z
        cp.add.at(numerator, r_i, W_data[:, None] * Z_rows)

    # Multiplicative KL update (matching your dense version structure)
    A_new = A * (numerator / (denominator + 1e-12))
    A_new = cp.clip(A_new, a_min=epsilon, a_max=None)
    return A_new


In [106]:
mode = 0
start_time = time.time()
factor_nodense_bis = kl_factor_update_nodense_equiv_bis(
                    vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    mode=mode,
                    shape=shape,)
print(f"calculated in {time.time()-start_time} seconds")


calculated in 1.9770660400390625 seconds


In [107]:
for el in [factor, factor_nodense, factor_nodense_bis]:
    for comp in [factor, factor_nodense, factor_nodense_bis]:
        print(cp.allclose(el, comp))

True


ValueError: operands could not be broadcast together with shapes (1000, 150) (8000, 150) () () ()

# Core calculations

In [125]:
from tensormet.sparse_ops import(
    gather_dense_at_block_nz,
    sparse_multi_mode_dot_vec
)

In [126]:
def kl_core_update(vec_tensor, shape, core, factors, modes, thread_budget, epsilon=1e-12):
    """
    One multiplicative KL update for the core tensor.

    Mirrors the sequence:
      - build dense reconstruction on CPU via ptl
      - gather reconstructed values at nonzero blocks
      - form X/R sparse ratio tensor
      - compute X_R = (X/R) ×_n W_n^T
      - compute F from column sums of factors
      - core *= X_R / F
      - normalize

    Returns
    -------
    core : updated core.
    """
    # Build a CPU Tucker object for pytensorlab reconstruction
    init_time = time.time()
    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]

    tucker = ptl.TuckerTensor(core=core_np, factors=factors_np)
    start_time = time.time()
    print(f"prepared env in {start_time-init_time} seconds")
    with thread_budget.limit():
        R = ptl_tucker_to_tensor(tucker)
    R_time = time.time()
    print(f"built R in {R_time-start_time} seconds")
    # Gather reconstructed values at nonzero coordinates of vec_tensor
    data = gather_dense_at_block_nz(R, vec_tensor, shape)
    data = tl.clip(data, a_min=epsilon, a_max=None)
    data_time = time.time()
    print(f"prepared data in {data_time-R_time} seconds")
    # X/R at nonzeros
    X_R_data = vec_tensor.data / cp.asarray(data)
    X_R = cpx_sparse.coo_matrix(
        (X_R_data, (vec_tensor.row, vec_tensor.col)),
        shape=vec_tensor.shape,
    )
    X_R_time = time.time()
    print(f"calculated X_R in {X_R_time-data_time} seconds")

    # (X/R) ×_n W_n^T   -> core-shaped tensor
    X_R = sparse_multi_mode_dot_vec(
        vec_tensor=X_R,
        orig_shape=shape,
        factors=factors,
        modes=modes,
        transpose_factors=True,
    )
    X_R = tl.clip(X_R, a_min=epsilon, a_max=None)
    X_R_prep_time = time.time()
    print(f"prepared X_R in {X_R_prep_time-X_R_time} seconds")
    # F = outer product of column sums of factors, broadcast to core shape
    col_sums = [tl.sum(A_n, axis=0) for A_n in factors]
    F = col_sums[0].reshape((core.shape[0],) + (1,) * (core.ndim - 1))
    for n in range(1, core.ndim):
        shape_n = [1] * n + [core.shape[n]] + [1] * (core.ndim - n - 1)
        F = F * col_sums[n].reshape(tuple(shape_n))
    F = tl.clip(F, a_min=epsilon, a_max=None)
    print(f"calculated F in {time.time() - start_time} seconds")
    # Multiplicative core update
    core *= X_R / (F + epsilon)
    return core


In [88]:
core_old = kl_core_update(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,)

prepared env in 0.00814509391784668 seconds


MemoryError: Unable to allocate 805. GiB for an array with shape (6000, 6000, 6000) and data type float32

In [127]:
import cupy as cp
import numpy as np
import cupyx
import cupyx.scipy.sparse as cpx_sparse

def _blocked_coo_to_flat_indices(vec_tensor, orig_shape):
    """
    vec_tensor is a COO matrix built from a blocked linearization:
        flat = row + col * block_size
    where block_size = min(prod(shape), int32_max).

    Returns
    -------
    flat : cupy.ndarray (nnz,) int64
    vals : cupy.ndarray (nnz,) float
    """
    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    coo = vec_tensor.tocoo()
    # Ensure int64 to avoid overflow
    flat = coo.row.astype(cp.int64) + coo.col.astype(cp.int64) * cp.int64(block_size)
    vals = coo.data
    return flat, vals
def _unravel_flat_indices_C(flat, shape):
    """
    flat : (m,) cupy int64
    shape: tuple of dims (I0, I1, ..., I_{N-1})

    Returns
    -------
    idxs : list of cupy arrays, each (m,)
           idxs[n] are indices along mode n.
    """
    shape = tuple(int(s) for s in shape)
    N = len(shape)
    u = flat
    idxs_rev = []
    for dim in reversed(shape):        # last mode fastest
        dim = int(dim)
        idxs_rev.append(u % dim)
        u = u // dim
    return list(reversed(idxs_rev))
def _einsum_letters(n):
    letters = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")
    if n > len(letters):
        raise ValueError(f"Tensor order {n} too large ({n}).")
    return letters[:n]


In [128]:
import cupy as cp
import numpy as np

def _rhat_from_factor_rows_sequential(core, mats, epsilon=1e-12):
    """
    core: (R0, R1, ..., R_{N-1})
    mats[n]: (b, Rn) factor rows for each mode at the b coordinates

    Returns
    -------
    r_hat : (b,)
    """
    N = core.ndim
    b = mats[0].shape[0]

    # Start by contracting mode 0 to introduce batch dimension:
    # tmp[b, R1, R2, ...] = sum_{r0} mats0[b,r0] * core[r0, R1, ...]
    tmp = cp.tensordot(mats[0], core, axes=(1, 0))  # (b, R1, R2, ..., R_{N-1})

    # Then fold in remaining modes with multiply+sum over the next rank axis each time.
    for n in range(1, N):
        # tmp has shape (b, Rn, R_{n+1}, ...)
        # multiply by mats[n] broadcasted onto axis=1, then sum over axis=1
        shp = (b, mats[n].shape[1]) + (1,) * (tmp.ndim - 2)
        tmp = cp.sum(tmp * mats[n].reshape(shp), axis=1)

    r_hat = cp.clip(tmp, a_min=epsilon, a_max=None)  # (b,)
    return r_hat
def _accumulate_core_num_outer(Num, w, mats):
    """
    Num: core-shaped accumulator (R0,...,R_{N-1})
    w  : (b,)
    mats[n]: (b, Rn)

    Updates Num in-place: Num += sum_m w[m] * outer(mats0[m], mats1[m], ...)
    """
    b = w.shape[0]
    N = len(mats)

    # Build T with shape (b, R0, R1, ..., R_{N-1}) using progressive outer products
    T = w[:, None] * mats[0]  # (b, R0)

    for n in range(1, N):
        # Expand last axis and outer with mats[n]
        # T: (b, ..., R_{n-1}) -> (b, ..., R_{n-1}, 1)
        # mats[n]: (b, Rn) -> (b, 1, ..., 1, Rn)
        T = T[..., None] * mats[n].reshape((b,) + (1,) * (T.ndim - 1) + (mats[n].shape[1],))

    # Sum over batch and add to Num
    Num += cp.sum(T, axis=0)


In [129]:
from tqdm import tqdm
def _blocked_coo_to_flat_indices(vec_tensor, orig_shape):
    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    coo = vec_tensor.tocoo()
    flat = coo.row.astype(cp.int64) + coo.col.astype(cp.int64) * cp.int64(block_size)
    vals = coo.data
    return flat, vals

def _unravel_flat_indices_C(flat, shape):
    shape = tuple(int(s) for s in shape)
    u = flat
    idxs_rev = []
    for dim in reversed(shape):
        idxs_rev.append(u % dim)
        u = u // dim
    return list(reversed(idxs_rev))

@profile
def kl_core_update_norecon_v2(
    vec_tensor,
    shape,
    core,
    factors,
    modes=None,              # assumes all modes
    thread_budget=None,      # kept for API compatibility
    epsilon=1e-12,
    batch_rhat=500_000,
    batch_num=32,
):
    init_time = time.time()
    shape = tuple(int(s) for s in shape)
    N = len(shape)
    if modes is None:
        modes = list(range(N))
    if list(modes) != list(range(N)):
        raise NotImplementedError("This version assumes modes == all modes (0..N-1).")
    start_time = time.time()
    print(f"init in {start_time - init_time}")
    flat, xvals = _blocked_coo_to_flat_indices(vec_tensor, shape)
    nnz = int(flat.size)
    if nnz == 0:
        return core

    nnz_time = time.time()

    print(f"nnz calcul in {nnz_time - start_time}")

    idxs = _unravel_flat_indices_C(flat, shape)  # list length N, each (nnz,)

    # Denominator is outer product of column sums, but don't materialize F.
    idx_time = time.time()
    print(f"ids calcul in {idx_time - start_time}")
    sums = [cp.clip(cp.sum(factors[n], axis=0), a_min=epsilon, a_max=None) for n in range(N)]

    Num = cp.zeros_like(core)

    # --- Pass 1: compute w = x / r_hat in big batches, stash w (or stream into pass 2)
    # Stashing w costs nnz floats; if that's too big, you can stream (see note below).
    w_all = cp.empty_like(xvals)

    before_batch_time = time.time()
    for start in range(0, nnz, int(batch_rhat)):
        end = min(start + int(batch_rhat), nnz)

        mats = [factors[n][idxs[n][start:end]] for n in range(N)]  # each (b, Rn)
        r_hat = _rhat_from_factor_rows_sequential(core, mats, epsilon=epsilon)  # (b,)
        w_all[start:end] = xvals[start:end] / r_hat
    after_batch_time = time.time()
    print(f"batch rhat calcul in {after_batch_time - before_batch_time}")
    # --- Pass 2: accumulate numerator in tiny batches (controls peak memory)
    for start in tqdm(range(0, nnz, int(batch_num))):
        end = min(start + int(batch_num), nnz)
        mats = [factors[n][idxs[n][start:end]] for n in range(N)]
        w = w_all[start:end]
        _accumulate_core_num_outer(Num, w, mats)

    accumulate_time = time.time()
    print(f"accumulate calcul in {accumulate_time - after_batch_time}")
    # --- MU update: core *= Num / (outer product of sums)
    core_new = core * (Num + epsilon)  # keep >0

    # Divide by sums via broadcasting, no F allocation
    for n in range(N):
        shp = [1] * N
        shp[n] = sums[n].shape[0]
        core_new = core_new / sums[n].reshape(tuple(shp))
    print(f"finalize core calcul in {time.time() - accumulate_time}")
    core_new = cp.clip(core_new, a_min=epsilon, a_max=None)
    return core_new


In [92]:
core_new = kl_core_update_norecon_v2(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,
                    batch_rhat=25_000,
                    batch_num=64,
                    )

ERROR: Could not find file /tmp/ipykernel_2719205/4247976739.py
init in 2.7179718017578125e-05
nnz calcul in 0.0006418228149414062
ids calcul in 0.0009686946868896484
batch rhat calcul in 0.08955979347229004


  4%|▎         | 1344/38176 [00:05<02:22, 259.32it/s]


KeyboardInterrupt: 

In [139]:
import cupy as cp
import numpy as np

def _gpu_free_bytes():
    """
    Conservative 'free bytes now' estimate.
    - memGetInfo: free bytes in the CUDA context (driver view)
    - default mempool: may have cached blocks; it can still grow if driver has free
    We return driver-free (most conservative for new allocations).
    """
    free_b, total_b = cp.cuda.runtime.memGetInfo()
    return int(free_b)


def estimate_batch_num_for_outer(
    factors,
    core,
    safety=0.50,
    temp_mult=1.5,
):
    """
    Much more conservative batch estimate for _accumulate_core_num_outer,
    because it often materializes intermediate outer-products / broadcasts.

    This is intentionally pessimistic: uses prod(R) scaling as a worst-case.
    """
    N = len(factors)
    dtype = core.dtype

    itemsize = int(np.dtype(dtype).itemsize)
    R = [int(factors[n].shape[1]) for n in range(N)]
    core_size = int(np.prod(R))

    # worst-case: per-b element touches/temporaries proportional to core_size
    # (many implementations end up with something like (b, core_size) transiently)
    bytes_per_b = core_size * itemsize

    # plus gathered mats and indices (usually small compared to core_size, but add anyway)
    bytes_per_b += sum(R) * itemsize + N * 8
    bytes_per_b = int(np.ceil(bytes_per_b * temp_mult))

    free_b = _gpu_free_bytes()
    budget_b = int(free_b * safety)

    b = max(1, budget_b // max(1, bytes_per_b))
    return int(b)


def estimate_batch_rhat_for_tensordot(
    core,
    factors,
    safety=0.60,
    temp_mult=1.3,     # tensordot may use extra workspace; keep >1
):
    """
    Estimate safe batch size for _rhat_from_factor_rows_sequential that starts with:
        tmp = tensordot(mats[0], core, axes=(1,0))  -> (b, prod(R[1:]))
    """
    N = len(factors)
    R = [int(factors[n].shape[1]) for n in range(N)]

    dtype = core.dtype
    itemsize = int(np.dtype(dtype).itemsize)

    # Dominant allocation: tmp has (b, prod(R[1:])) elements
    prod_rest = int(np.prod(R[1:])) if N > 1 else 1
    tmp_bytes_per_b = prod_rest * itemsize

    # Also gather mats: sum_n (b*Rn) elements
    mats_bytes_per_b = sum(R) * itemsize

    # Small vectors (r_hat, w) and idx slices (negligible, but include)
    small_bytes_per_b = (4 * itemsize) + (N * 8)

    bytes_per_b = int(np.ceil((tmp_bytes_per_b + mats_bytes_per_b + small_bytes_per_b) * temp_mult))

    free_b = _gpu_free_bytes()
    budget_b = int(free_b * safety)

    b = budget_b // max(1, bytes_per_b)
    return int(b)


In [145]:
# pick batch sizes based on current free memory
batch_num  = estimate_batch_num_for_outer(factors, core, safety=0.99, temp_mult=1.1)
batch_rhat_bis = estimate_batch_rhat_for_tensordot(core, factors, safety=0.99, temp_mult=1.1)
print("batch_rhat bis:", batch_rhat_bis)

print("auto batch_num:", batch_num)

batch_rhat bis: 5490
auto batch_num: 11


In [147]:
core_new = kl_core_update_norecon_v2(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,
                    batch_rhat=batch_rhat_bis*10,
                    batch_num=batch_num*10,
                    )

ERROR: Could not find file /tmp/ipykernel_2719205/4247976739.py
init in 2.193450927734375e-05
nnz calcul in 0.0007512569427490234
ids calcul in 0.0010290145874023438
batch rhat calcul in 30.922117710113525


  0%|          | 0/23743 [00:08<?, ?it/s]


OutOfMemoryError: Out of memory allocating 55,000,000,000 bytes (allocated so far: 10,336,821,760 bytes).

In [23]:
cp.allclose(core_old, core_new)

array(True)

In [114]:
import cupy as cp
import numpy as np
import cupyx.scipy.sparse as cpx_sparse

# --- reuse your helpers (or keep these definitions if not already in scope) ---

def _blocked_coo_to_flat_indices(vec_tensor, orig_shape):
    """
    vec_tensor is a COO matrix built from a blocked linearization:
        flat = row + col * block_size
    where block_size = min(prod(shape), int32_max).
    """
    orig_shape = tuple(orig_shape)
    size = int(np.prod(orig_shape))
    int32_max = np.iinfo(np.int32).max
    block_size = min(size, int32_max)

    coo = vec_tensor.tocoo()
    flat = coo.row.astype(cp.int64) + coo.col.astype(cp.int64) * cp.int64(block_size)
    vals = coo.data
    return flat, vals


def _unravel_flat_indices_C(flat, shape):
    """C-order unravel: last mode varies fastest."""
    shape = tuple(int(s) for s in shape)
    u = flat
    idxs_rev = []
    for dim in reversed(shape):
        idxs_rev.append(u % dim)
        u = u // dim
    return list(reversed(idxs_rev))


def _rhat_from_factor_rows_sequential(core, mats, epsilon=1e-12):
    """
    core: (R0, R1, ..., R_{N-1})
    mats[n]: (b, Rn) factor rows for each mode at the b coordinates
    returns r_hat: (b,)
    """
    N = core.ndim
    b = mats[0].shape[0]

    tmp = cp.tensordot(mats[0], core, axes=(1, 0))  # (b, R1, R2, ...)
    for n in range(1, N):
        shp = (b, mats[n].shape[1]) + (1,) * (tmp.ndim - 2)
        tmp = cp.sum(tmp * mats[n].reshape(shp), axis=1)

    return cp.clip(tmp, a_min=epsilon, a_max=None)


def _einsum_letters(n):
    letters = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")
    if n > len(letters):
        raise ValueError(f"Tensor order {n} too large ({n}).")
    return letters[:n]


def _tucker_sum_all_entries(core, factors, epsilon=1e-12):
    """
    Exact sum(R) where R = Tucker(core, factors), without forming R.

    sum_R = sum_{r0..rN-1} core[r0..rN-1] * Π_n s_n[rn]
    where s_n[rn] = sum_i A^{(n)}[i, rn]
    """
    N = core.ndim
    letters = _einsum_letters(N)
    core_subs = "".join(letters)

    sums = [cp.sum(factors[n], axis=0) for n in range(N)]
    sums = [cp.clip(s, a_min=epsilon, a_max=None) for s in sums]

    # eq: "abc,a,b,c->" (for N=3), etc.
    eq = core_subs + "," + ",".join(letters) + "->"
    sum_R = cp.einsum(eq, core, *sums)
    return cp.clip(sum_R, a_min=epsilon, a_max=None)


# --- no-dense KL error ---

def kl_compute_errors_norecon(
    vec_tensor: cpx_sparse.spmatrix,
    shape,
    core,
    factors,
    thread_budget=None,          # kept for API compatibility; unused
    epsilon=1e-12,
    batch_rhat=500_000,
):
    """
    Relative generalized KL divergence C_KL(X || R) for sparse X,
    WITHOUT forming dense R, staying close to the core-update approach.

    Computes:
      KL = sum_{nz} [x log(x/r) - x + r] + (sum_R - sum_{nz} r)
      rel_KL = KL / sum_{nz} x
    """
    shape = tuple(int(s) for s in shape)
    N = len(shape)

    flat, x_nz = _blocked_coo_to_flat_indices(vec_tensor, shape)
    nnz = int(flat.size)
    if nnz == 0:
        # If X is all-zeros, KL reduces to sum_R. Relative term is ill-defined; mirror your style:
        sum_R = _tucker_sum_all_entries(core, factors, epsilon=epsilon)
        return sum_R / cp.maximum(cp.asarray(0.0, dtype=sum_R.dtype), epsilon)

    x_nz = cp.asarray(x_nz)
    x_nz = cp.clip(x_nz, a_min=epsilon, a_max=None)

    idxs = _unravel_flat_indices_C(flat, shape)  # list of N arrays, each (nnz,)

    # --- compute r_nz in batches (like your core update r_hat pass) ---
    r_nz = cp.empty_like(x_nz)
    for start in range(0, nnz, int(batch_rhat)):
        end = min(start + int(batch_rhat), nnz)
        mats = [factors[n][idxs[n][start:end]] for n in range(N)]  # each (b, Rn)
        r_nz[start:end] = _rhat_from_factor_rows_sequential(core, mats, epsilon=epsilon)

    r_nz = cp.clip(r_nz, a_min=epsilon, a_max=None)

    # --- KL contribution from nonzeros ---
    term_pos = x_nz * cp.log(x_nz / r_nz) - x_nz + r_nz
    kl_pos = cp.sum(term_pos)

    # --- zero contribution: sum_R - sum_{nz} r_nz ---
    sum_R = _tucker_sum_all_entries(core, factors, epsilon=epsilon)
    sum_R_nz = cp.sum(r_nz)
    kl_zero = sum_R - sum_R_nz

    kl_total = kl_pos + kl_zero

    sum_X = cp.sum(x_nz)
    rel_kl = kl_total / cp.maximum(sum_X, epsilon)
    return rel_kl


In [115]:
def kl_compute_errors(
        vec_tensor: cpx_sparse.spmatrix,
        shape,
        core,
        factors,
        thread_budget: ThreadBudget,
        epsilon=1e-12,
):

    """Generalised KL divergence C_KL(X || R) for sparse X.

    vec_tensor : vectorised sparse X (block-encoded, same as 'tensor')
    shape      : original N-D shape
    core       : current core G
    factors    : list of factor matrices A^{(n)}
    """

    core_np = tl.to_numpy(core)
    factors_np = [tl.to_numpy(f) for f in factors]

    tucker = ptl.TuckerTensor(core=core_np,
                                  factors=factors_np)
    with thread_budget.limit():
        R = ptl_tucker_to_tensor(tucker)

    shape = tuple(shape)

    # --- 1) Dense reconstruction R = G ×_1 A^{(1)} × ... ×_N A^{(N)} ---
    # This is exactly step 3 in Table 2 of the paper.
    # This breaks with dims over 1000
    # R = tucker_to_tensor((core, factors))      # cp.ndarray, shape=shape
    # R = tl.clip(R, a_min=epsilon, a_max=None)
    # R_flat = R.ravel()                         # length = size

    r_nz = gather_dense_at_block_nz(R, vec_tensor, shape)
    r_nz = tl.clip(r_nz, a_min=epsilon, a_max=None)
    # --- 2) Decode sparse X indices to flat indices ---
    X_coo = vec_tensor.tocoo()
    x_nz = X_coo.data

    # --- 3) X_i and R_i at nonzero entries ---
    # the original data can still contain harmful zeros
    x_nz = tl.clip(x_nz, a_min=epsilon, a_max=None)
    r_nz = cp.asarray(r_nz)

    # --- 4) KL contribution from nonzeros ---
    # sum_{i: X_i>0} [X_i log(X_i/R_i) - X_i + R_i]
    term_pos = x_nz * cp.log(x_nz / r_nz) - x_nz + r_nz
    kl_pos = cp.sum(term_pos)
    # --- 5) KL contribution from zero entries ---
    # For X_i = 0, the KL term tends to R_i (limit X→0).
    # Full sum over all i is:
    #   ∑_i [X_i log(X_i/R_i) - X_i + R_i]
    # = (sum over nonzeros) + (sum over zeros),
    # and sum_{zeros} R_i = sum(R) - sum_{nonzeros} R_i.
    sum_R = cp.sum(R)
    sum_R_nz = cp.sum(r_nz)
    kl_zero = sum_R - sum_R_nz
    kl_total = kl_pos + kl_zero

    # --- 6) Optional normalized "relative" KL error ---
    sum_X = cp.sum(x_nz)  # sum over nonzero X
    rel_kl = kl_total / cp.maximum(sum_X, epsilon)
    # print(f"KL divergence: {kl_total}, relative KL: {rel_kl}")
    return rel_kl

In [116]:
kl_compute_errors_norecon(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    shape=shape,
                    thread_budget=thread_budget,
                    batch_rhat=batch_rhat_bis,
                          )

array(1.1516349e+11, dtype=float32)

In [27]:
kl_compute_errors(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    shape=shape,
                    thread_budget=thread_budget,)

array(7.299757, dtype=float32)

In [18]:
fast_core = kl_core_update_fast(vec_tensor=sparse_tensor.tensor,
                    core=core,
                    factors=factors,
                    modes=modes,
                    shape=shape,
                    thread_budget=thread_budget,)

/tmp/ipykernel_2697491/3097603159.py:24: SyntaxWarning: invalid escape sequence '\h'
  Compute reconstruction \hat x_i at a batch of indices i (given as idxs per mode).


OutOfMemoryError: Out of memory allocating 2,700,000,000,000 bytes (allocated so far: 18,978,018,304 bytes).